# Train, quantize and export a 4 → 4 → 4 network

Run the cells in order, or use **Restart Kernel and Run All**. This notebook is self-contained; it does not load `npu_quantization.py` or require variables from old notebook versions.

- Training: PyTorch, ReLU hidden layer, four output logits.
- Quantization: symmetric INT8 weights/activations, zero point 0, INT32 biases in accumulator units.
- Requantization: positive INT32 multiplier `M`, shift `S` in 0..62, nearest rounding with ties to even, INT8 clamp.
- Calibration: fixed scales from the training set, including a **fixed final output scale**.
- Export: `trained_npu_params.h`, plus preprocessing metadata and raw-input reference vectors.

**The C NPU still needs the planned multiplier/shift implementation and matching rounding before it can match these predictions. Its current shift-only block is incompatible with this export.**

Dependencies: `numpy`, `torch`, `scikit-learn`. The dataset has two target classes; the requested network still has four logits. Outputs 2 and 3 have no corresponding training target. This is a machine-learning development example, not a clinical prediction tool.

In [1]:
# Cell 1: Load data and fit preprocessing on TRAINING data only.
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
FEATURE_GAIN = 25.0
feature_indices = [0, 2, 3, 7]
data = load_breast_cancer()
X_selected = data.data[:, feature_indices]
feature_names = [str(data.feature_names[i]) for i in feature_indices]
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_selected, data.target, test_size=0.2, random_state=SEED, stratify=data.target
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw) * FEATURE_GAIN
X_test = scaler.transform(X_test_raw) * FEATURE_GAIN

# Optional host preprocessing of the four ORIGINAL feature measurements.
def preprocess_raw_features(values):
    values = np.asarray(values, dtype=np.float64)
    single = values.ndim == 1
    if single:
        values = values[None, :]
    if values.ndim != 2 or values.shape[1] != 4 or not np.isfinite(values).all():
        raise ValueError("Expected four finite original feature measurements")
    transformed = scaler.transform(values) * FEATURE_GAIN
    return transformed[0] if single else transformed

print("Features in input order:", feature_names)
print("Classes:", dict(enumerate(data.target_names.tolist())))
print(f"Train samples: {len(X_train)}, test samples: {len(X_test)}")
print(f"Training model-input range: [{X_train.min():.3f}, {X_train.max():.3f}]")

Features in input order: ['mean radius', 'mean perimeter', 'mean area', 'mean concave points']
Classes: {0: 'malignant', 1: 'benign'}
Train samples: 455, test samples: 114
Training model-input range: [-50.681, 134.260]


In [2]:
# ============================================================
# Cell 2: Define and Train the 4->4->4 Network
# ============================================================
class Net444(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 4)
        self.fc2 = nn.Linear(4, 4)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)  # raw logits, no output activation
        return x

torch.manual_seed(SEED)
model = Net444()

X_tr = torch.tensor(X_train, dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.long)
X_te = torch.tensor(X_test, dtype=torch.float32)
y_te = torch.tensor(y_test, dtype=torch.long)

optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

for epoch in range(300):
    model.train()
    optimizer.zero_grad()
    out = model(X_tr)
    loss = criterion(out, y_tr)
    loss.backward()
    optimizer.step()

# Evaluate
model.eval()
with torch.no_grad():
    logits = model(X_te)
    preds = logits.argmax(dim=1)
    acc = (preds == y_te).float().mean().item()

print(f"Training complete. Test accuracy (float32): {acc*100:.2f}%")
print(f"Final loss: {loss.item():.4f}")

Training complete. Test accuracy (float32): 91.23%
Final loss: 0.1892


## Quantization and integer reference
Biases are quantized directly from their original float values as `round(b / (input_scale * weight_scale))`. There are no separate INT8 bias arrays or bias scales.

`inference_int8(real_values, params)` quantizes **already preprocessed model-input floats** at the host boundary. `inference_raw_int8(int_values, params)` accepts **raw INT8 values**, exactly like the current C TB. Both use the same fixed per-layer parameters. Integer inference uses INT64 temporaries to check INT32 partial sums and safely multiply by `M`; it does not use float dot products.

In [3]:
# Cell 3: Shared quantization, inference and C-export functions.
import math
import numpy as np

INT32_MIN = -(1 << 31)
INT32_MAX = (1 << 31) - 1


def as_numpy(value):
    if hasattr(value, "detach"):
        value = value.detach().cpu().numpy()
    return np.asarray(value)


def activation_scale(values):
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0 or not np.isfinite(values).all():
        raise ValueError("Calibration needs nonempty finite data")
    maximum = float(np.max(np.abs(values)))
    return maximum / 127.0 if maximum else 1.0


def quantize_int8(values, scale):
    # Explicit rounding rule: nearest, ties to EVEN (also used by torch.round).
    values = as_numpy(values).astype(np.float64)
    if not math.isfinite(scale) or scale <= 0 or not np.isfinite(values).all():
        raise ValueError("Inputs must be finite and scale must be positive")
    return np.clip(np.rint(values / scale), -128, 127).astype(np.int8)


def quantize_multiplier(real_multiplier):
    """Positive INT32 M and S in [0,62]: real_multiplier ~= M / 2**S."""
    if not math.isfinite(real_multiplier) or real_multiplier <= 0:
        raise ValueError("Requantization multiplier must be finite and positive")
    fraction, exponent = math.frexp(real_multiplier)
    multiplier = int(round(fraction * (1 << 31)))
    if multiplier == (1 << 31):
        multiplier >>= 1
        exponent += 1
    shift = 31 - exponent
    if not 0 <= shift <= 62:
        raise ValueError("Scale ratio is outside this M/S implementation's range")
    return multiplier, shift


def rounded_right_shift(values, shift):
    """Signed integer division by 2**S; nearest with ties to EVEN.

    Input is an INT32 accumulator times positive INT32 M, so abs(input)
    fits INT64. Quotient/remainder avoid overflow from adding a rounding bias.
    C must implement this rule for negative values as well as positive ones.
    """
    if not 0 <= shift <= 62:
        raise ValueError("Shift must be in [0,62]")
    values = np.asarray(values, dtype=np.int64)
    if np.any(values == np.iinfo(np.int64).min):
        raise OverflowError("INT64_MIN is outside the supported product range")
    if shift == 0:
        return values.copy()
    magnitude = np.abs(values)
    divisor = np.int64(1 << shift)
    quotient, remainder = np.divmod(magnitude, divisor)
    half = divisor // 2
    increment = (remainder > half) | ((remainder == half) & ((quotient & 1) != 0))
    rounded = quotient + increment.astype(np.int64)
    return np.where(values < 0, -rounded, rounded)


def requantize(accumulator, multiplier, shift):
    accumulator = np.asarray(accumulator, dtype=np.int64)
    if np.any(accumulator < INT32_MIN) or np.any(accumulator > INT32_MAX):
        raise OverflowError("Accumulator exceeds INT32")
    if not 1 <= int(multiplier) <= INT32_MAX:
        raise ValueError("Multiplier must be positive signed INT32")
    product = accumulator * np.int64(multiplier)  # Safe INT64 product.
    rounded = rounded_right_shift(product, shift)
    return np.clip(rounded, -128, 127).astype(np.int8)


def prepare_quantized_model(model, calibration_inputs):
    """Export a trained Net444 using ONLY training/calibration inputs.

    W shape is [output neuron, input feature], matching the NPU's rows.
    Calibrate input, post-ReLU hidden, and final output scales ONCE.
    Calibration uses a float64 evaluation of the trained float32 parameters.
    """
    x = as_numpy(calibration_inputs).astype(np.float64)
    weights = [as_numpy(layer.weight).astype(np.float64)
               for layer in (model.fc1, model.fc2)]
    biases = [as_numpy(layer.bias).astype(np.float64)
              for layer in (model.fc1, model.fc2)]
    if x.ndim != 2 or x.shape[1] != 4 or x.shape[0] == 0:
        raise ValueError("Calibration inputs must have shape [samples,4]")
    for w, b in zip(weights, biases):
        if w.shape != (4, 4) or b.shape != (4,) or not np.isfinite(b).all():
            raise ValueError("Expected finite 4x4 weights and length-4 biases")
    hidden = np.maximum(x @ weights[0].T + biases[0], 0)
    logits = hidden @ weights[1].T + biases[1]
    scales = [activation_scale(x), activation_scale(hidden), activation_scale(logits)]
    layers = []
    for index, (w, b) in enumerate(zip(weights, biases)):
        weight_scale = activation_scale(w)
        accumulator_scale = scales[index] * weight_scale
        w_q = quantize_int8(w, weight_scale)
        # IMPORTANT: quantize ORIGINAL float biases directly into MAC units.
        # Do not quantize biases to INT8 first and do not use separate b_scale.
        b_rounded = np.rint(b / accumulator_scale)
        if not np.isfinite(b_rounded).all() or np.any(b_rounded < INT32_MIN) or np.any(b_rounded > INT32_MAX):
            raise OverflowError("Bias does not fit INT32 in accumulator units")
        b_q = b_rounded.astype(np.int32)
        # Conservative bound: every intermediate fits for ANY signed INT8 x.
        bound = np.abs(b_q.astype(np.int64)) + 128 * np.abs(w_q.astype(np.int64)).sum(axis=1)
        if np.any(bound > INT32_MAX):
            raise OverflowError("Bias/weights cannot guarantee safe INT32 accumulation")
        real_multiplier = accumulator_scale / scales[index + 1]
        multiplier, shift = quantize_multiplier(real_multiplier)
        layers.append(dict(weights=w_q, biases=b_q, multiplier=multiplier,
                           shift=shift, relu=(index == 0), weight_scale=weight_scale,
                           accumulator_scale=accumulator_scale, real_multiplier=real_multiplier))
    return dict(layers=layers, input_scale=scales[0], hidden_scale=scales[1], output_scale=scales[2])


def mac_int32(x_q, weights, biases):
    # Integer arithmetic only. INT64 temporaries detect overflow rather than
    # silently wrapping; each checked result represents an INT32 register.
    x = x_q.astype(np.int64)
    acc = np.broadcast_to(biases.astype(np.int64), (x.shape[0], 4)).copy()
    for column in range(4):
        acc += x[:, column, None] * weights[None, :, column].astype(np.int64)
        if np.any(acc < INT32_MIN) or np.any(acc > INT32_MAX):
            raise OverflowError("MAC partial sum exceeds INT32")
    return acc.astype(np.int32)


def inference_raw_int8(x_q, params):
    """Same input convention as the C TB: already-quantized INT8 values."""
    x = as_numpy(x_q)
    if x.dtype.kind not in "iu" or np.any(x < -128) or np.any(x > 127):
        raise ValueError("Raw input must contain integers in [-128,127]")
    single = x.ndim == 1
    if single:
        x = x[None, :]
    if x.ndim != 2 or x.shape[1] != 4:
        raise ValueError("Input must have shape [4] or [samples,4]")
    x = x.astype(np.int8)
    for layer in params["layers"]:
        acc = mac_int32(x, layer["weights"], layer["biases"])
        if layer["relu"]:
            acc = np.maximum(acc, 0)
        x = requantize(acc, layer["multiplier"], layer["shift"])
    return x[0] if single else x


def inference_int8(x_float, params):
    """Host quantizes real features, then runs the integer-only reference.

    Returns dequantized output, output INT8 values, and FIXED output scale.
    Float conversion occurs only at the host input/output boundaries.
    """
    x_q = quantize_int8(x_float, params["input_scale"])
    out_q = inference_raw_int8(x_q, params)
    return out_q.astype(np.float64) * params["output_scale"], out_q, params["output_scale"]


def export_c_header(params, filename="trained_npu_params.h"):
    """C arrays for SRAM preloading; M/S/ReLU must also be loaded by the NPU."""
    def initializer(value):
        if isinstance(value, (list, tuple, np.ndarray)):
            return "{" + ", ".join(initializer(v) for v in value) + "}"
        return str(int(value))
    layers = params["layers"]
    lines = ["#ifndef TRAINED_NPU_PARAMS_H", "#define TRAINED_NPU_PARAMS_H", "#include <stdint.h>",
             "/* W[output][input]; INT32 biases are already in accumulator units.",
             " * Requantize: clamp_int8(round_ties_even((int64_t)acc * M / 2^S)).",
             " * Configuration SRAM addresses/packing are defined by the C model.",
             " * Float scales below are HOST metadata, not required in the MAC. */"]
    for key in ("input_scale", "hidden_scale", "output_scale"):
        lines.append(f"#define NPU_{key.upper()} {params[key]:.17g}")
    declarations = [("int8_t", "trained_weights[2][4][4]", [l["weights"] for l in layers]),
                    ("int32_t", "trained_biases[2][4]", [l["biases"] for l in layers]),
                    ("int32_t", "requant_multiplier[2]", [l["multiplier"] for l in layers]),
                    ("uint32_t", "requant_shift[2]", [l["shift"] for l in layers]),
                    ("uint32_t", "relu_enable[2]", [l["relu"] for l in layers])]
    for dtype, name, values in declarations:
        lines.append(f"static const {dtype} {name} = {initializer(values)};")
    lines.append("#endif")
    with open(filename, "w", encoding="utf-8") as output:
        output.write("\n".join(lines) + "\n")
    print(f"Exported {filename}")

In [4]:
# Cell 4: Prepare parameters ONCE and evaluate with the integer reference.
params = prepare_quantized_model(model, X_tr)
print("Fixed activation scales:")
for name in ("input_scale", "hidden_scale", "output_scale"):
    print(f"  {name}: {params[name]:.12g}")
for index, layer in enumerate(params["layers"], 1):
    print(f"\nLayer {index}: ReLU={layer['relu']}, M={layer['multiplier']}, S={layer['shift']}")
    print(f"  Weight scale: {layer['weight_scale']:.12g}")
    print(f"  Accumulator scale: {layer['accumulator_scale']:.12g}")
    print("  Weights [output][input]:\n", layer["weights"])
    print("  INT32 biases:", layer["biases"].tolist())

output_real, output_q, fixed_output_scale = inference_int8(X_te, params)
integer_predictions = output_q.argmax(axis=1)
integer_accuracy = np.mean(integer_predictions == y_test)
print(f"\nFloat32 accuracy: {acc * 100:.2f}%")
print(f"Integer-reference accuracy (argmax INT8): {integer_accuracy * 100:.2f}%")
print("INT8 rounding/clipping can create ties and change predictions.")
print("Output scale is FIXED across all samples:", fixed_output_scale)

Fixed activation scales:
  input_scale: 1.05716843492
  hidden_scale: 1.14702404837
  output_scale: 2.35745326372

Layer 1: ReLU=True, M=1585768005, S=38
  Weight scale: 0.00625933436897
  Accumulator scale: 0.00661717071851
  Weights [output][input]:
 [[  19   25  -57   43]
 [ -51  -37 -127  -46]
 [ 111   -9   69   50]
 [  75   21   78   25]]
  INT32 biases: [42, 124, 183, -54]

Layer 2: ReLU=False, M=1143168641, S=38
  Weight scale: 0.00854753790878
  Accumulator scale: 0.00980423153575
  Weights [output][input]:
 [[   3   26    1   63]
 [ -62   38   30   17]
 [ -65 -121 -121 -127]
 [  17  -40  -45   10]]
  INT32 biases: [-13, 112, -60, -72]

Float32 accuracy: 91.23%
Integer-reference accuracy (argmax INT8): 90.35%
INT8 rounding/clipping can create ties and change predictions.
Output scale is FIXED across all samples: 2.357453263723774


## Export the full numerical contract
The header contains `trained_weights`, `trained_biases`, `requant_multiplier`, `requant_shift`, and `relu_enable`. Copying weights and biases alone will **not** configure requantization. The C controller/fetcher must also load M/S/ReLU from SRAM as planned.

`npu_preprocessing.json` records input scaling and the four-feature preprocessing for host-side use. `npu_reference_vectors.json` contains raw INT8 vectors and their expected results for later C comparison. No runtime output scale is recalculated per input.

In [5]:
# Cell 5: Export everything from params; no stale w1_q/b1_scale variables.
import json
from pathlib import Path

export_c_header(params, "trained_npu_params.h")
print(Path("trained_npu_params.h").read_text())
metadata = {
    "feature_indices": feature_indices,
    "feature_names": feature_names,
    "standard_scaler_mean": scaler.mean_.tolist(),
    "standard_scaler_scale": scaler.scale_.tolist(),
    "feature_gain": FEATURE_GAIN,
    "input_scale": params["input_scale"],
    "hidden_scale": params["hidden_scale"],
    "output_scale": params["output_scale"],
    "zero_point": 0,
    "rounding": "nearest_ties_to_even",
    "shift_range": [0, 62],
    "input_formula": "q = clamp_int8(round(((features-mean)/std * feature_gain) / input_scale))",
    "output_formula": "approximate_logit = output_int8 * output_scale",
    "weight_layout": "W[layer][output][input]",
    "class_names": data.target_names.tolist(),
}
Path("npu_preprocessing.json").write_text(json.dumps(metadata, indent=2) + "\n")
reference_input = np.array([
    [1, 2, 3, 4], [0, 0, 0, 0], [-128, -128, -128, -128],
    [127, 127, 127, 127], [-128, 127, -128, 127]
], dtype=np.int16)
reference_output = inference_raw_int8(reference_input, params)
reference_vectors = [
    {"input_q": x.tolist(), "expected_output_q": y.tolist()}
    for x, y in zip(reference_input, reference_output)
]
Path("npu_reference_vectors.json").write_text(json.dumps(reference_vectors, indent=2) + "\n")
print("Also exported npu_preprocessing.json and npu_reference_vectors.json")
print("C scale block must implement M/S and ties-to-even rounding before comparison.")

Exported trained_npu_params.h
#ifndef TRAINED_NPU_PARAMS_H
#define TRAINED_NPU_PARAMS_H
#include <stdint.h>
/* W[output][input]; INT32 biases are already in accumulator units.
 * Requantize: clamp_int8(round_ties_even((int64_t)acc * M / 2^S)).
 * Configuration SRAM addresses/packing are defined by the C model.
 * Float scales below are HOST metadata, not required in the MAC. */
#define NPU_INPUT_SCALE 1.057168434924028
#define NPU_HIDDEN_SCALE 1.147024048373249
#define NPU_OUTPUT_SCALE 2.357453263723774
static const int8_t trained_weights[2][4][4] = {{{19, 25, -57, 43}, {-51, -37, -127, -46}, {111, -9, 69, 50}, {75, 21, 78, 25}}, {{3, 26, 1, 63}, {-62, 38, 30, 17}, {-65, -121, -121, -127}, {17, -40, -45, 10}}};
static const int32_t trained_biases[2][4] = {{42, 124, 183, -54}, {-13, 112, -60, -72}};
static const int32_t requant_multiplier[2] = {1585768005, 1143168641};
static const uint32_t requant_shift[2] = {38, 38};
static const uint32_t relu_enable[2] = {1, 0};
#endif

Also exported

## Manual inference
Edit `user_input` and `input_mode`, then rerun the next cell.

- **`raw_int8`**: four integers in [-128,127]. Use this mode to compare exactly against the C TB input.
- **`model_float`**: four model-input floats *after* StandardScaler and the ×25 gain. These are quantized before inference.
- **`features`**: four original dataset feature measurements in the printed feature order. Host preprocessing and input quantization are both applied.

Every mode prints the actual `input_q` to enter into the C TB. The output vector is fixed-scale INT8, not the old per-vector normalized display.

In [6]:
# Cell 6: Manual inference. Rerun after editing these two values.
user_input = [1, 2, 3, 4]
input_mode = "raw_int8"  # "raw_int8", "model_float", or "features"


def show_inference(values, mode="raw_int8", number=1):
    if mode == "raw_int8":
        x_q = np.asarray(values)
        # The integer engine validates type, shape and range before any cast.
        out_q = inference_raw_int8(x_q, params)
    elif mode in ("model_float", "features"):
        x_float = (preprocess_raw_features(values) if mode == "features"
                   else np.asarray(values, dtype=np.float64))
        x_q = quantize_int8(x_float, params["input_scale"])
        out_q = inference_raw_int8(x_q, params)
    else:
        raise ValueError("Mode must be raw_int8, model_float, or features")
    if x_q.shape != (4,):
        raise ValueError("Manual inference requires exactly one length-4 input vector")
    output_float = out_q.astype(np.float64) * params["output_scale"]
    prediction = int(out_q.argmax())
    class_name = (str(data.target_names[prediction]) if prediction < len(data.target_names)
                  else "unused output: no corresponding training target")
    print("=" * 60)
    print(f"Inference #{number}: RESULT | input_mode={mode}")
    print("Entered input:", list(values))
    print("Raw INT8 input for C TB:", x_q.tolist())
    print("Predicted output (quantized INT8):", out_q.tolist())
    print("Approximate dequantized logits:", output_float.tolist())
    print("Fixed output scale:", params["output_scale"])
    print(f"Argmax output index: {prediction} ({class_name})")
    print("=" * 60)
    return out_q


manual_inference_number = globals().get("manual_inference_number", 0) + 1
manual_output = show_inference(user_input, input_mode, manual_inference_number)

Inference #1: RESULT | input_mode=raw_int8
Entered input: [1, 2, 3, 4]
Raw INT8 input for C TB: [1, 2, 3, 4]
Predicted output (quantized INT8): [0, 1, -4, -1]
Approximate dequantized logits: [0.0, 2.357453263723774, -9.429813054895096, -2.357453263723774]
Fixed output scale: 2.357453263723774
Argmax output index: 1 (benign)


In [7]:
# Cell 7: Optional terminal-style loop; not started by Run All.
# After the notebook is ready, call interactive_inference("raw_int8").
def interactive_inference(mode="raw_int8"):
    if mode not in ("raw_int8", "model_float", "features"):
        raise ValueError("Mode must be raw_int8, model_float, or features")
    number = 1
    while True:
        try:
            text = input(f"Enter 4 values ({mode}), or q to quit: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nFinished.")
            return
        if text.lower() == "q":
            return
        try:
            tokens = text.replace(",", " ").split()
            if len(tokens) != 4:
                raise ValueError("Exactly four values are required")
            values = [int(v) for v in tokens] if mode == "raw_int8" else [float(v) for v in tokens]
            show_inference(values, mode, number)
        except (ValueError, OverflowError) as error:
            print("Invalid input:", error)
            continue
        number += 1

# Uncomment ONLY when you want the blocking interactive loop:
# interactive_inference("raw_int8")